# 10.3 上下文压缩 (Context Compression)

上下文压缩减少长序列的显存和计算开销，使模型能处理更长的上下文。

本节涵盖：
- 滑动窗口注意力
- KV缓存压缩
- 分块处理

## 1. 滑动窗口注意力

**目的**：限制每个token的注意力范围

**基本原理**：每个token只关注固定窗口大小内的前序token，KV缓存只需保留最近w个token的KV，显存从O(n)降至O(w)。

**信息流动**：虽然单层只能看到w个token，但多层堆叠后信息可以逐层传递到更远的位置，感受野随层数增长。

**代表**：Mistral（滑动窗口8K）、Longformer

In [ ]:
import torch

torch.manual_seed(42)

seq_len = 128
window_size = 32
n_layers = 4

print('=== Sliding Window Attention ===')
print(f'Sequence: {seq_len}, Window: {window_size}, Layers: {n_layers}')
print(f'\nKV cache per layer: {window_size} tokens (not {seq_len})')
print(f'KV cache reduction: {window_size/seq_len:.1%} of full attention')
print(f'\nReceptive field growth through layers:')
for layer in range(n_layers):
    rf = min(seq_len, (layer + 1) * window_size)
    print(f'  Layer {layer}: receptive field = {rf} tokens')
print(f'\nKey: Sliding window limits KV cache to O(w) per layer.')
print(f'Information flows across windows through layer stacking.')

## 2. KV缓存压缩与分块处理

**KV缓存压缩**：保留重要的KV对，丢弃不重要的，常用策略：
- **H2O**：基于注意力分数保留最重要的KV
- **StreamingLLM**：保留attention sink（前几个token）+ 最近窗口

**分块处理**：将长序列分成多个chunk分别处理，再聚合结果：
- **Chunked Attention**：每个chunk内部完整注意力，chunk间稀疏连接
- **Hierarchical Attention**：底层处理chunk，高层处理chunk摘要

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

seq_len = 1024
n_heads = 4
d_head = 32
n_keep = 64
n_sink = 4

attn_weights = torch.softmax(torch.randn(1, n_heads, seq_len, seq_len) / d_head**0.5, dim=-1)
importance = attn_weights.sum(dim=(0, 1, 2))
_, top_indices = importance.topk(n_keep)
sink_indices = torch.arange(n_sink)
keep_indices = torch.cat([sink_indices, top_indices[top_indices >= n_sink][:n_keep - n_sink]])

print('=== KV Cache Compression ===')
print(f'Full KV cache: {seq_len} tokens')
print(f'Compressed: {n_keep} tokens ({n_keep/seq_len:.1%})')
print(f'  Attention sinks: first {n_sink} tokens (always kept)')
print(f'  Important tokens: top-{n_keep-n_sink} by attention score')

chunk_size = 256
n_chunks = seq_len // chunk_size
print(f'\n=== Chunked Processing ===')
print(f'Chunk size: {chunk_size}, Chunks: {n_chunks}')
print(f'Per-chunk attention: O({chunk_size}²) = {chunk_size**2/1e6:.2f}M')
print(f'Full attention: O({seq_len}²) = {seq_len**2/1e6:.2f}M')
print(f'Savings: {n_chunks} chunks x {chunk_size**2/1e6:.2f}M = {n_chunks*chunk_size**2/1e6:.2f}M')
print(f'\nKey: Compression trades quality for memory efficiency.')
print(f'StreamingLLM enables infinite-length generation with fixed memory.')